# **Sequential Decision-Making and $Q$-Learning**

**Lecture (60 min) + lab (60 min)**  
**Setup:** Python 3; no external packages required.

By the end, you can describe an MDP, distinguish a policy from a value function, and train a small Q-learning agent in a GridWorld.

| Time | Activity |
|---|---|
| 0–15 min | States, actions, rewards, and transitions |
| 15–25 min | Policies, values, and exploration |
| 25–45 min | Q-learning update |
| 45–60 min | Run and interpret a tiny GridWorld |
| 60–120 min | Lab: teach the agent to navigate obstacles |

## 1. Sequential decisions as an MDP

A **Markov decision process (MDP)** models an agent acting over time:

- **State** `s`: what the agent currently knows about the environment.
- **Action** `a`: a choice available in that state.
- **Transition**: the next state after an action (possibly uncertain).
- **Reward** `r`: immediate feedback for the transition.
- **Discount** `gamma`: how much future rewards matter compared with immediate reward.

The **Markov property** says the current state contains the information needed to predict what happens next. A **policy** chooses actions from states. A **value function** estimates how much future reward a state (or state-action pair) will yield.

## 2. Q-learning: learn by trying

Q-learning estimates `Q(s, a)`, the expected long-run reward for taking action `a` in state `s` and then behaving well. Its one-step update is:

`Q(s,a) <- Q(s,a) + alpha * (r + gamma * max_a' Q(s',a') - Q(s,a))`

`alpha` is the learning rate. The max term bootstraps from the next state. During training, **epsilon-greedy exploration** takes a random action with probability `epsilon`; otherwise it chooses the action with the largest current Q-value. Exploration helps discover routes; exploitation uses what has been learned.

In [ ]:
import random

SIZE = 4
START = (0, 0)
GOAL = (3, 3)
ACTIONS = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # right, down, left, up

def step(state, action):
    row, col = state
    dr, dc = action
    next_state = (max(0, min(SIZE - 1, row + dr)), max(0, min(SIZE - 1, col + dc)))
    done = next_state == GOAL
    reward = 10 if done else -1
    return next_state, reward, done

q = {(r, c): [0.0] * len(ACTIONS) for r in range(SIZE) for c in range(SIZE)}
print("States:", len(q), "| actions per state:", len(ACTIONS))

In [ ]:
alpha, gamma, epsilon = 0.2, 0.95, 0.2

for episode in range(500):
    state = START
    for _ in range(30):
        if random.random() < epsilon:
            action_index = random.randrange(len(ACTIONS))
        else:
            action_index = max(range(len(ACTIONS)), key=lambda i: q[state][i])
        next_state, reward, done = step(state, ACTIONS[action_index])
        target = reward if done else reward + gamma * max(q[next_state])
        q[state][action_index] += alpha * (target - q[state][action_index])
        state = next_state
        if done:
            break

def greedy_route():
    state, route = START, [START]
    for _ in range(12):
        if state == GOAL:
            break
        action_index = max(range(len(ACTIONS)), key=lambda i: q[state][i])
        state, _, _ = step(state, ACTIONS[action_index])
        route.append(state)
    return route

route = greedy_route()
print("Greedy route:", route)
print("Reached goal:", route[-1] == GOAL)

### Read the result

The agent receives `-1` for every non-terminal move and `+10` for reaching the goal, so short successful routes are valuable. Training is not a pre-written route: the Q-values are updated from the agent's own experience. Re-run the cell to see how random exploration can affect the route.

**Check:** Why is the update target just `reward` when `done` is true? What might happen if epsilon were zero before the agent found the goal?

## Lab (60 min): navigate around obstacles

Modify the environment to include blocked cells at `(1, 1)` and `(2, 1)`. The agent must not enter a blocked cell. A blocked move should leave the agent in its current state and still cost `-1`.

### Tasks

1. Update `step` to reject blocked moves while preserving the grid boundaries.
2. Train the agent again and show a greedy route from start to goal that avoids both blocked cells.
3. Compare training with `epsilon = 0.2` and `epsilon = 0.0`. Run each setting several times; record whether the goal was reached and explain the difference.
4. Change the step reward to `-2`. Does the learned route or number of steps change? Explain what the reward is encouraging.

**Suggested timing:** environment change (10 min), train/test (25 min), compare settings (15 min), interpretation (10 min).

In [ ]:
BLOCKED = {(1, 1), (2, 1)}

# TODO: replace this function with obstacle-aware transitions.
def step_with_obstacles(state, action):
    return step(state, action)

# TODO: train using step_with_obstacles, then print a greedy route.
# The route must reach GOAL and must not contain a blocked cell.

### Exit ticket

Submit your modified transition function, training code, route, and a short comparison of exploration versus exploitation. **Extension:** make one action occasionally move in a random direction and discuss why the learned policy may need more training.